# Lab 3: Quality Evaluation - Accuracy Testing & AgentCore Evaluations

Lab 2 showed you how fast and how expensive your agents are. This lab answers the harder question: are they actually producing correct results?

A fast, cheap agent that gives wrong answers is worse than useless. And the right evaluation strategy depends on the type of output — exact matching for structured data, subjective scoring for free-form text.

You'll use two complementary approaches:

| Approach | What It Measures | When to Use |
|----------|------------------|-------------|
| **Programmatic (Part 1)** | Precision, Recall, F1 against ground truth | Structured outputs with known correct answers |
| **AgentCore Evaluations (Part 2)** | LLM-as-Judge quality metrics | Open-ended responses, multi-turn sessions, tool accuracy |

**Prerequisites:** Complete Lab 1 and Lab 2

**Time:** 40 minutes

---
## Part 1: Programmatic Accuracy Testing

Programmatic testing compares agent outputs against known correct answers (ground truth). This works well for your Lab 1 agents because they produce structured outputs.

### Understanding Precision, Recall, and F1

| Metric | Formula | What It Means for Claims |
|--------|---------|-------------------------|
| **Precision** | TP / (TP + FP) | Of items agent identified, how many were correct? |
| **Recall** | TP / (TP + FN) | Of items that should be identified, how many did agent find? |
| **F1 Score** | 2 × (P × R) / (P + R) | Harmonic mean - balances precision and recall - How many did the agent identified without being frequently incorrect? |

Example: If your `inspection` agent flags 10 claims as fraudulent:
- 8 are actually fraudulent (True Positives)
- 2 are legitimate (False Positives)  
- 3 fraudulent claims were missed (False Negatives)

Then: Precision = 8/10 = 80%, Recall = 8/11 = 73%, F1 = 76%

### Step 1.1: Create Ground Truth Dataset

Ground truth is the "correct answer" for each test case — the reference your agent's outputs will be compared against. Without it, you can't say whether an agent is right, only whether it finished.

For claims processing, ground truth captures what a human expert would say: which documents support the claim, what risk level the incident represents, whether coverage applies, and which fraud patterns (if any) are present.

A good ground truth dataset is **small, representative, and versioned alongside your code** — treat it the same as tests. Three claims is enough to demonstrate the pattern; production systems typically use 50-500 labeled examples curated from real cases.


In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Set
import statistics
from claims_agents import ClaimsAssistant, create_test_claims, parse_structured_tail, node_text

@dataclass
class GroundTruth:
    """Expected correct outputs for a claim."""
    claim_id: str
    expected_documents: Set[str]
    expected_risk_level: str  # low, medium, high
    expected_coverage: bool
    expected_fraud_flags: Set[str]

GROUND_TRUTH = {
    'claim_001': GroundTruth(
        claim_id='claim_001',
        expected_documents={'accident_report.pdf', 'damage_photos.jpg'},
        expected_risk_level='medium',
        expected_coverage=True,
        expected_fraud_flags=set()
    ),
    'claim_002': GroundTruth(
        claim_id='claim_002',
        expected_documents={'plumber_report.pdf', 'water_damage_photos.jpg'},
        expected_risk_level='low',
        expected_coverage=True,
        expected_fraud_flags=set()
    ),
    'claim_003': GroundTruth(
        claim_id='claim_003',
        expected_documents={'police_report.pdf', 'garage_security_footage.mp4'},
        expected_risk_level='high',
        expected_coverage=True,
        expected_fraud_flags={'high_value', 'theft'}
    ),
}

print(f"Ground truth defined for {len(GROUND_TRUTH)} claims")

### Step 1.2: Calculate Precision, Recall, F1

Document identification and fraud-flag detection are **set comparison** problems — you produce a set of findings, ground truth has a set of expected findings. Three standard metrics quantify how well the sets match:

| Metric | What it measures | Formula |
|---|---|---|
| **Precision** | Of what the agent flagged, how much was correct? | TP / (TP + FP) |
| **Recall** | Of what should have been flagged, how much did the agent catch? | TP / (TP + FN) |
| **F1** | Balanced score combining both | 2·P·R / (P+R) |

In claims processing, **recall matters more for fraud detection** (missing fraud is expensive) while **precision matters more for document extraction** (false positives waste reviewer time). F1 is a safe single-number summary.


In [ ]:
def calculate_set_metrics(predicted: Set[str], expected: Set[str]) -> Dict[str, float]:
    """Calculate precision, recall, F1 for set-based predictions."""
    if not predicted and not expected:
        return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
    
    tp = len(predicted & expected)
    fp = len(predicted - expected)
    fn = len(expected - predicted)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {'precision': precision, 'recall': recall, 'f1': f1}

def calculate_exact_match(predicted: str, expected: str) -> float:
    """Calculate exact match for categorical predictions."""
    return 1.0 if predicted.lower() == expected.lower() else 0.0

# Test
test_metrics = calculate_set_metrics({'a', 'b', 'c'}, {'a', 'b', 'd'})
print(f"Test: P={test_metrics['precision']:.2f}, R={test_metrics['recall']:.2f}, F1={test_metrics['f1']:.2f}")

### Step 1.3: Run Evaluation Against Ground Truth

Run each labeled claim through your Lab 1 `ClaimsAssistant`, then compare the agent's outputs against ground truth using the metrics you just defined.

Each agent response ends with a `STRUCTURED_OUTPUT` JSON block (enforced via the system prompt in `claims_agents.py`). The evaluator parses that block to extract comparable fields — `identified_documents`, `risk_level`, `fraud_flags` — without needing to interpret the conversational text.

This is the "programmatic" part of evaluation: deterministic, fast, CI-friendly. You can run it on every PR to catch regressions.


In [ ]:
def evaluate_claim_result(result, ground_truth: GroundTruth) -> Dict[str, any]:
    """Evaluate a single claim result against ground truth.

    Each agent response ends with a STRUCTURED_OUTPUT JSON block that we parse
    to extract comparable fields (see claims_agents.py).
    """
    nodes = result.results
    scores = {"claim_id": ground_truth.claim_id}

    doc = parse_structured_tail(node_text(nodes.get("document_analysis")))
    predicted_docs = set(doc.get("identified_documents", []))
    scores["document_f1"] = calculate_set_metrics(
        predicted_docs, ground_truth.expected_documents
    )["f1"]
    scores["_predicted_docs"] = predicted_docs

    insp = parse_structured_tail(node_text(nodes.get("inspection")))
    predicted_risk = insp.get("risk_level", "unknown")
    scores["risk_accuracy"] = calculate_exact_match(
        predicted_risk, ground_truth.expected_risk_level
    )
    scores["_predicted_risk"] = predicted_risk
    predicted_flags = set(insp.get("fraud_flags", []))
    scores["fraud_f1"] = calculate_set_metrics(
        predicted_flags, ground_truth.expected_fraud_flags
    )["f1"]
    scores["_predicted_flags"] = predicted_flags

    return scores

# Run evaluation (suppress the agents' verbose reasoning; we can focus on the scores here)
import contextlib, io

assistant = ClaimsAssistant()
test_claims = create_test_claims()
evaluation_results = []

print("Evaluating claims against ground truth...\n")
for claim in test_claims:
    if claim.claim_id in GROUND_TRUTH:
        with contextlib.redirect_stdout(io.StringIO()):
            result = assistant.analyze_claim(claim)
        scores = evaluate_claim_result(result, GROUND_TRUTH[claim.claim_id])
        evaluation_results.append(scores)
        gt = GROUND_TRUTH[claim.claim_id]
        print(f"\n{claim.claim_id}")
        print(f"  Doc F1={scores['document_f1']:.2f}  Risk={scores['risk_accuracy']:.2f}  Fraud F1={scores['fraud_f1']:.2f}")
        if scores['risk_accuracy'] < 1.0:
            print(f"    risk:  predicted={scores['_predicted_risk']!r}  expected={gt.expected_risk_level!r}")
        if scores['fraud_f1'] < 1.0:
            print(f"    flags: predicted={scores['_predicted_flags']}  expected={gt.expected_fraud_flags}")
        if scores['document_f1'] < 1.0:
            print(f"    docs:  predicted={scores['_predicted_docs']}  expected={gt.expected_documents}")


### Step 1.4: Aggregate Accuracy Metrics

Per-claim scores tell you how any single case performed. Aggregate scores across your test set tell you whether the system is good enough to ship.

Look at the aggregates three ways:

- **Mean F1 across claims** — overall accuracy. Set a quality gate (e.g. ≥ 0.80).
- **Worst-case F1** — the hardest claim in your set. If it's below your minimum, fix that case first.
- **Per-field breakdown** — which field is failing? Risk level might score well while fraud flags lag, telling you which agent prompt needs work.


In [ ]:
def aggregate_metrics(results: List[Dict]) -> Dict[str, float]:
    metrics = {}
    for key in ['document_f1', 'risk_accuracy', 'fraud_f1']:
        values = [r[key] for r in results if key in r]
        if values:
            metrics[f'{key}_mean'] = statistics.mean(values)
    return metrics

aggregate = aggregate_metrics(evaluation_results)

print("=" * 50)
print("PROGRAMMATIC ACCURACY SUMMARY")
print("=" * 50)
print(f"Document Identification F1: {aggregate.get('document_f1_mean', 0):.2%}")
print(f"Risk Assessment Accuracy:   {aggregate.get('risk_accuracy_mean', 0):.2%}")
print(f"Fraud Detection F1:         {aggregate.get('fraud_f1_mean', 0):.2%}")

---
## Part 2: AgentCore Evaluations (LLM-as-Judge)

Programmatic metrics work for structured outputs, but your `claim_summary` agent produces free-form text. AgentCore Evaluations uses LLM-as-Judge to assess quality.

### Why AgentCore Evaluations?

| Programmatic Testing | AgentCore Evaluations |
|---------------------|----------------------|
| Binary right/wrong | Nuanced scoring (1-5 scale) |
| Requires exact ground truth | Can evaluate without ground truth |
| Single response only | Session, Trace, and Tool-call levels |
| No tool accuracy | ToolSelectionAccuracy, ParameterAccuracy |

### Evaluation Levels

| Level | Scope | Example for Claims |
|-------|-------|-------------------|
| **Session** | Multi-turn conversation | Did full workflow achieve goal? |
| **Trace** | Single interaction | Was this claim analysis correct? |
| **Tool Call** | Individual function | Did agent pick right tool? |

### Step 2.1: Setup AgentCore Evaluation Client

Programmatic F1 scores are great for structured outputs, but they can't score free-form text like the claim summary agent's final narrative. For that you need an **LLM-as-a-Judge** — a second model that rates the agent's output against a rubric.

AgentCore Evaluations provides a managed LLM-as-a-Judge service with 13 built-in evaluators (helpfulness, correctness, harmfulness, etc.) plus custom evaluators you define. Install and initialize its client below.


In [ ]:
# Install workshop dependencies. These pins are the versions the workshop was tested on.

!pip install --quiet --upgrade \
    'strands-agents[otel]==1.15.1' \
    'bedrock-agentcore-starter-toolkit>=0.1,<1.0' \
    'aws-opentelemetry-distro>=0.10,<1.0' \
    'pydantic>=2.10,<3.0' \
    'boto3>=1.35,<2.0' \
    'awscli>=1.35,<2.0' \
    'opentelemetry-api>=1.30,<2.0'


In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation
from boto3.session import Session
import json

boto_session = Session()
region = boto_session.region_name or 'us-east-1'

eval_client = Evaluation(region=region)
print(f"AgentCore Evaluation client initialized in {region}")

### Step 2.2: Explore Built-in Evaluators

AgentCore provides 13 pre-configured evaluators organized by what they measure:

| Category | Evaluators | Scope |
|---|---|---|
| **Response Quality** | Correctness, Completeness, Helpfulness, Coherence | TRACE |
| **Task Completion** | GoalSuccess | SESSION |
| **Tool Usage** | ToolSelectionAccuracy, ParameterAccuracy | TOOL_CALL |
| **Safety** | Harmfulness, Stereotyping, PII leakage | TRACE |

**Levels matter**: a SESSION evaluator sees the whole conversation, a TRACE evaluator sees one turn, a TOOL_CALL evaluator sees one invocation. Pick the level that matches what you're testing.

List them below to see the full catalog.


In [ ]:
# List available evaluators
available_evaluators = eval_client.list_evaluators()

print("AgentCore Built-in Evaluators:")
print("=" * 50)
for evaluator in available_evaluators.get('evaluators', []):
    print(f"  {evaluator['evaluatorId']}")

# Get details on Correctness
print("\nBuiltin.Correctness details:")
correctness = eval_client.get_evaluator(evaluator_id="Builtin.Correctness")
print(f"  Description: {correctness.get('description', 'N/A')}")

### Step 2.3: Create Custom Evaluator for Claims

Built-in evaluators handle generic quality dimensions. But "is this a good insurance claim analysis?" requires domain knowledge — you need to check for claim type identification, risk appropriateness, coverage accuracy, and actionable next steps.

Custom evaluators let you define a domain-specific rubric. AgentCore runs a judge model (configurable — we use Claude Haiku 4.5 here) against your rubric to score every trace. You control:

- **Rating scale** — numerical (0-5) or categorical
- **Instructions** — what the judge should evaluate and how
- **Judge model** — any Bedrock model; smaller/cheaper for high-volume eval


In [ ]:
claims_evaluator_config = {
    "llmAsAJudge": {
        "modelConfig": {
            "bedrockEvaluatorModelConfig": {
                "modelId": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
                "inferenceConfig": {
                    "maxTokens": 500,
                    "temperature": 0.0
                }
            }
        },
        "ratingScale": {
            "numerical": [
                {"value": 1.0, "label": "Very Poor", "definition": "Fails to address the claim"},
                {"value": 2.0, "label": "Poor", "definition": "Multiple criteria missed"},
                {"value": 3.0, "label": "Adequate", "definition": "Basic criteria met"},
                {"value": 4.0, "label": "Good", "definition": "Most criteria met"},
                {"value": 5.0, "label": "Excellent", "definition": "All criteria met"}
            ]
        },
        "instructions": """You are evaluating an insurance claims processing agent.

Context: {context}
Target to evaluate: {assistant_turn}

Evaluate on these criteria:
1. CLAIM IDENTIFICATION: Did the agent correctly identify claim type and key details?
2. RISK ASSESSMENT: Is the risk level appropriate?
3. COVERAGE ANALYSIS: Did the agent correctly determine policy coverage?
4. ACTIONABILITY: Are recommended next steps clear and actionable?

Rate the overall response quality on the 1-5 scale."""
    }
}

# Create the evaluator
custom_evaluator = eval_client.create_evaluator(
    name="claims_processing_quality",
    level="TRACE",
    description="Custom evaluator for insurance claims processing",
    config=claims_evaluator_config
)

evaluator_id = custom_evaluator['evaluatorId']
print(f"Created custom evaluator: {evaluator_id}")


### Step 2.4: Map Evaluators to Lab 1 Agents

Not every evaluator applies to every agent. A document-extraction agent should be scored on completeness and correctness; a summary agent on helpfulness and coherence; an inspection agent on correctness and harmfulness.

Mapping evaluators to agents keeps evaluation signal high and noise low — you won't penalise the document extractor for not being "helpful" in a conversational sense.

The dictionary below is the mapping you'll use in the next cell when invoking evaluations. In production you'd apply it in the online evaluation config so every live session gets the right scores automatically.


In [ ]:
AGENT_EVALUATION_MAPPING = {
    'document_analysis': ['Builtin.Completeness', 'Builtin.Correctness'],
    'policy_retrieval': ['Builtin.Correctness'],
    'inspection': ['Builtin.Correctness', 'Builtin.Harmfulness'],
    'claim_summary': ['Builtin.Completeness', 'Builtin.Helpfulness', 'Builtin.Coherence'],
    'full_workflow': ['Builtin.GoalSuccess', evaluator_id]
}

print("Evaluation mapping for Lab 1 agents:")
for agent, evaluators in AGENT_EVALUATION_MAPPING.items():
    print(f"  {agent}: {evaluators}")

### Step 2.5: Deploy the Agent to AgentCore Runtime

AgentCore evaluators run against **sessions** captured by AgentCore observability. To produce those sessions we deploy `ClaimsAssistant` (from `claims_agents.py`) behind the AgentCore Runtime using the entrypoint in `agentcore_entrypoint.py`.

Deployment takes a few minutes — CodeBuild builds an ARM64 container in the cloud, then launches it. The `agentcore` CLI was installed earlier in this notebook.


In [ ]:
import os, subprocess, shlex

REGION = os.environ.get('AWS_REGION', 'us-east-1')
AGENT_NAME = 'claims_assistant'

def sh(cmd, quiet=False):
    """Run a shell command. If quiet, suppress stdout unless the command fails."""
    r = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
    if not quiet:
        print(r.stdout)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(f'command failed: {cmd}')
    return r.stdout

# Configure (quiet — prints a lot of diagnostic output)
print("Configuring agent...")
sh(f'agentcore configure --entrypoint agentcore_entrypoint.py '
   f'--name {AGENT_NAME} --requirements-file requirements.txt '
   f'--region {REGION} --disable-memory --non-interactive', quiet=True)
print("  ✓ configured")

# Launch (let CodeBuild progress stream through — takes several minutes)
print("\nLaunching to AgentCore Runtime (this takes 3-5 minutes)...")
sh(f'agentcore launch --agent {AGENT_NAME} --auto-update-on-conflict')
print('Deployment complete.')


### Step 2.6: Generate Sessions and Run Evaluations

Invoke the deployed agent once per test claim to create AgentCore sessions with observability traces, then run a built-in evaluator against the most recent session.

CloudWatch ingestion has a 2-5 minute delay before sessions become evaluable. The cell waits, then runs `Builtin.Helpfulness`.


In [ ]:
import json, time, yaml, pathlib
from claims_agents import create_test_claims

# Invoke the deployed agent for each test claim (generates traced sessions).
# Agent responses are long; suppress them and show a one-line status per claim.
import subprocess as _subprocess
print("Invoking deployed agent for each test claim...")
for claim in create_test_claims():
    payload = json.dumps({'claim_id': claim.claim_id})
    result = _subprocess.run(
        ["agentcore", "invoke", "--agent", AGENT_NAME, payload],
        capture_output=True, text=True,
    )
    status = "ok" if result.returncode == 0 else "FAILED"
    print(f"  {claim.claim_id}: {status}")
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"invoke failed for {claim.claim_id}")

# Resolve the real AgentCore agent ID from the config written by `agentcore configure`
cfg_path = pathlib.Path('.bedrock_agentcore.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
agents = cfg.get('agents', {}) or {}
agent_cfg = agents.get(AGENT_NAME) or next(iter(agents.values()), {})
agent_runtime_id = (
    agent_cfg.get('bedrock_agentcore', {}).get('agent_id')
    or agent_cfg.get('agent_id')
)
if not agent_runtime_id:
    raise RuntimeError('Could not find deployed agent id in .bedrock_agentcore.yaml')
print(f'Resolved AgentCore agent id: {agent_runtime_id}')

print('\nWaiting 3 minutes for CloudWatch traces to populate...')
time.sleep(180)

# Run evaluators and capture results as JSON for a clean summary
results_path = pathlib.Path('eval_results.json')
sh(f'agentcore eval run --agent-id {agent_runtime_id} '
   f'--evaluator Builtin.Helpfulness --evaluator {evaluator_id} '
   f' --output {results_path}')

# Summarise the JSON output (schema: EvaluationResults.to_dict from the toolkit)
data = json.loads(results_path.read_text())
summary = data.get('summary', {})
session_id = data.get('session_id', 'unknown')

print('\n' + '=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
print(f"Session:    {session_id}")
print(f"Evaluators: {summary.get('total_evaluations', 0)} run, "
      f"{summary.get('successful', 0)} successful, "
      f"{summary.get('failed', 0)} failed")

for r in data.get('results', []):
    name = r.get('evaluator_name') or r.get('evaluator_id')
    value = r.get('value')
    label = r.get('label')
    error = r.get('error')
    tokens = (r.get('token_usage') or {}).get('totalTokens')

    print(f"\n  {name}")
    if error:
        print(f"    Error: {error}")
        continue
    score_line = f"    Score:  {value}" if value is not None else "    Score:  (categorical)"
    if label:
        score_line += f"  ({label})"
    print(score_line)
    if tokens is not None:
        print(f"    Tokens: {tokens}")
    explanation = (r.get('explanation') or '').strip()
    if explanation:
        first = explanation.split('. ')[0][:200]
        print(f"    Why:    {first}{'...' if len(explanation) > 200 else ''}")

print('\n' + '=' * 60)
print('View full results in CloudWatch:')
print(f"  GenAI Observability: https://console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core/agents")
print(f"  Lab 2 dashboard:     https://console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards:name=GenAI-Claims-Processing")


### View evaluation results in CloudWatch

Two dashboards give you complementary views:

**1. GenAI Observability (Bedrock AgentCore)** — the built-in dashboard AWS provides automatically. Shows sessions, traces, per-agent token usage, and **your evaluation scores** side-by-side.

Click your `claims_assistant` agent, then the **Evaluations** tab to see `Builtin.Helpfulness` and your custom `claims_processing_quality` scores per trace.

**2. `GenAI-Claims-Processing` dashboard (from Lab 2)** — your custom operational view. Still running and collecting metrics from any new invocations.

- Console: **CloudWatch → Dashboards → GenAI-Claims-Processing**

### Step 2.7: Clean Up

The deployed AgentCore Runtime keeps running until destroyed. Run this cell when you're done to avoid ongoing costs.


In [ ]:
sh(f'agentcore destroy --agent {AGENT_NAME} --force --delete-ecr-repo')

print("\nYour dashboards remain available for review:")
print(f"  GenAI Observability:  https://console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core/agents")
print(f"  Operational (Lab 2):  https://console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards:name=GenAI-Claims-Processing")


---
## Workshop Complete

Over three labs you built, monitored, and evaluated a real multi-agent system on AWS.

| Lab | What you built | What it answers |
|-----|---------------|-----------------|
| **Lab 1** | Four specialized agents orchestrated by Strands GraphBuilder | *Can I build it?* |
| **Lab 2** | CloudWatch dashboard with per-agent tokens, latency, cache, cost | *How much does it cost? How is it operating?* |
| **Lab 3** | Programmatic F1 tests + AgentCore LLM-as-Judge evaluations | *Is it actually correct?* |

### What makes this production-ready

- **Observability by default** — every invocation emits traces, every trace is evaluable.
- **Evaluation is measurable, not vibes** — F1 scores and rubric-based LLM scores, not "looks good."
- **Operational and quality views live side-by-side** — in the same CloudWatch console your SRE team already uses.

### Use it

The pattern generalizes. Swap in your domain (support, finance, healthcare), keep the same three-layer approach: build with Strands, operate with CloudWatch, evaluate with AgentCore. The evaluation rubrics, ground truth fixtures, and dashboards are reusable assets — version them alongside your agent code.

Ship agents you can measure. Measure the ones you ship.
